# Sales JSON - Bronze ingestion

**Medallion role:** Bronze. This notebook incrementally lands JSON sales orders from the ADLS landing zone into an external Delta table while preserving the source payload and operational lineage. Storage access is expected to be provided by the workspace and Unity Catalog configuration; no credentials are embedded in the notebook.

In [0]:
dbutils.widgets.removeAll()

## Runtime configuration and storage boundaries

The widgets make the same notebook deployable to `dev` and `prod` and provide an auditable ingestion timestamp. Landing data, Auto Loader schema history, streaming checkpoints, and external Delta data use separate paths because each has a different lifecycle and recovery purpose.

In [0]:
from datetime import datetime, timezone

default_ingestion_timestamp = datetime.now(timezone.utc).isoformat()

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    default_ingestion_timestamp,
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()
ingestion_timestamp = dbutils.widgets.get("ingestion_timestamp").strip()

if environment not in ["dev", "prod"]:
    raise ValueError("Environment must be either 'dev' or 'prod'.")


config = {
    "dev": {
        "storage_account": "stcentralusjrdev",
        "catalog": "salesjson_dev"
    },
    "prod": {
        "storage_account": "stcentralusjrprod",
        "catalog": "salesjson_prod"
    }
}

env = config[environment]

storage_account = env["storage_account"]
catalog = env["catalog"]

bronze_table = f"{catalog}.bronze.orders_raw"

source_path = (
    f"abfss://landing@{storage_account}.dfs.core.windows.net/"
    "salesjson/incoming/"
)

schema_path = (
    f"abfss://streaming@{storage_account}.dfs.core.windows.net/"
    "salesjson/schemas/orders/"
)

checkpoint_path = (
    f"abfss://streaming@{storage_account}.dfs.core.windows.net/"
    "salesjson/checkpoints/bronze_orders/"
)

bronze_path = (
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
    "salesjson/bronze/orders_raw/"
)

print(f"Environment           : {environment}")
print(f"Ingestion timestamp   : {ingestion_timestamp}")
print(f"Catalog               : {catalog}")
print(f"Source                : {source_path}")
print(f"Schema location       : {schema_path}")
print(f"Checkpoint            : {checkpoint_path}")
print(f"Bronze table          : {bronze_table}")
print(f"Bronze path           : {bronze_path}")

## Incremental discovery and schema protection

Auto Loader uses managed file events to discover new JSON files efficiently. The schema location persists inference state, `addNewColumns` permits additive evolution, and the rescued-data column retains fields that cannot be parsed instead of silently discarding them.

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp
)


bronze_stream = (
    spark.readStream
        .format("cloudFiles")

        # Source format
        .option("cloudFiles.format", "json")

        # Persist Auto Loader schema information
        .option(
            "cloudFiles.schemaLocation",
            schema_path
        )

        # Automatically add newly detected columns
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )

        # Use the managed file-events cache configured
        # on the Unity Catalog external location
        .option(
            "cloudFiles.useManagedFileEvents",
            "true"
        )

        # Preserve fields that cannot be parsed
        .option(
            "rescuedDataColumn",
            "_rescued_data"
        )

        .load(source_path)
)

## Bronze lineage metadata

File name, full path, modification time, and ingestion time are added before persistence. These technical columns support traceability, replay analysis, and downstream deduplication without changing the source business fields.

In [0]:
bronze_enriched = (
    bronze_stream

    .withColumn(
        "source_file",
        col("_metadata.file_name")
    )

    .withColumn(
        "source_file_path",
        col("_metadata.file_path")
    )

    .withColumn(
        "source_file_modification_time",
        col("_metadata.file_modification_time")
    )

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

## External Delta registration and checkpointed write

The Unity Catalog table is registered over an explicit lakehouse path, so the data lifecycle remains external to the catalog object. The append stream runs with `availableNow`, evolves the Delta schema when new columns arrive, and uses a dedicated checkpoint to prevent already-processed files from being ingested again.

In [0]:


spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table}
    USING DELTA
    LOCATION '{bronze_path}'
""")

print(f"External Delta table registered: {bronze_table}")

In [0]:
bronze_query = (
    bronze_enriched.writeStream

        .format("delta")

        .outputMode("append")

        # Required for the target Delta table to evolve
        # when Auto Loader discovers new columns.
        .option(
            "mergeSchema",
            "true"
        )

        .option(
            "checkpointLocation",
            checkpoint_path
        )

        .trigger(
            availableNow=True
        )

        .toTable(
            bronze_table
        )
)

bronze_query.awaitTermination()

print("Bronze ingestion completed successfully.")